# Spanish IMDb Review Sentiment with a Bidirectional LSTM

This experiment uses the Spanish-translated IMDb 50K dataset to classify movie reviews as positive or negative. The translations are machine-generated and may contain errors. Labels: `0 = negative`, `1 = positive`.


## 1. Provide the project files and dataset

In Colab, upload this notebook, `train.py`, and the Kaggle CSV. Download the CSV from [the dataset page](https://www.kaggle.com/datasets/luisdiegofv97/imdb-dataset-of-50k-movie-reviews-spanish).


In [ ]:
!pip -q install tensorflow pandas scikit-learn matplotlib
import os, shutil
from google.colab import files
uploaded = files.upload()  # Select IMDB Dataset SPANISH.csv and train.py if not already present
os.makedirs('data', exist_ok=True)
for name in uploaded:
    if name.lower().endswith('.csv'):
        shutil.move(name, os.path.join('data', 'IMDB Dataset SPANISH.csv'))
print('Project files:', os.listdir('.'))
print('Data files:', os.listdir('data'))


## 2. Explore the Spanish reviews

We inspect the available columns, label balance, missing values, example reviews, and review lengths before fitting the model.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('data/IMDB Dataset SPANISH.csv')
display(df.head(3))
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
review_col = 'review_es' if 'review_es' in df.columns else 'review'
label_col = 'sentimiento' if 'sentimiento' in df.columns else 'sentiment'
print('Missing values:\n', df[[review_col, label_col]].isna().sum())
print('Label counts:\n', df[label_col].value_counts(dropna=False))
df['word_count'] = df[review_col].fillna('').astype(str).str.split().str.len()
plt.hist(df['word_count'].clip(upper=800), bins=40)
plt.title('Spanish IMDb review lengths (capped at 800 words)')
plt.xlabel('Words'); plt.ylabel('Reviews'); plt.show()


## 3. Clean, split, train, and evaluate

The shared `train.py` script removes missing/empty rows, maps Spanish or English label spellings to binary classes, creates stratified train/validation/test sets, fits the tokenizer on training text only, and trains a Bidirectional LSTM. The test set remains separate until final evaluation.


In [ ]:
import train
train.main()


## 4. Review the saved results

The script reports accuracy, precision, recall, F1, and per-class scores. It saves training curves, a confusion matrix, metrics, and ten held-out test predictions in `outputs/`. The Streamlit app uses the model and tokenizer saved under `models/`.


In [ ]:
from IPython.display import display, Image
display(Image('outputs/training_plot.png'))
display(Image('outputs/confusion_matrix.png'))
display(pd.read_csv('outputs/test_predictions.csv'))
